In [1]:
!pip install vllm==0.10.2

In [2]:
!pip install litellm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 27.5 MB/s eta 0:00:00


In [3]:
!nohup vllm serve Qwen/Qwen3-1.7B \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  > vllm.log &

nohup: redirecting stderr to stdout


In [7]:
!tail -n 20 vllm.log # to watch last 20 rows og vllm.log, about 2/3 minutes to have the server ready -> Application startup complete

(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /v1/chat/completions, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /v1/completions, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /v1/embeddings, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /pooling, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /classify, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /score, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /v1/score, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /v1/audio/transcriptions, Methods: POST
(APIServer pid=1970) INFO 11-26 09:19:09 [launcher.py:44] Route: /v1/audio/translations, Methods: POST
(APIServer pid=1970) INFO 11-

In [8]:
import sqlite3
db_name = "shakespeare.sqlite"
conn = sqlite3.connect(db_name)

In [9]:
import pandas as pd

def query_db(sql: str):
    """
    Executes an SQL query and returns the result as JSON

    Args:
        sql: The SQL query to execute

    Returns:
        The result of the query as JSON
    """
    try:
        df = pd.read_sql_query(sql, conn)
        return df.to_dict(orient="records")
    except Exception as e:
        return {"error": str(e)}

In [10]:
def get_schema(connection):
    schema_str = ""
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table_name in tables:
        table_name = table_name[0]
        if table_name.startswith("sqlite_"):
            continue

        cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}';")
        create_statement = cursor.fetchone()[0]
        schema_str += create_statement + ";\n"

    return schema_str

In [11]:
db_schema = get_schema(conn)

In [12]:
def get_function_by_name(name):
    if name == "query_db":
        return query_db

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "query_db",
            "description": "Executes a single SQL query on the database. Use this to make exploratory queries in order to find the final answer.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": 'The SQL query to execute on the database.',
                    }
                },
                "required": ["sql"],
            },
        },
    }
]

In [13]:
USER_QUESTION = f"""
Database Schema:
{db_schema}
Evidence:
King John refers to Title = 'King John'.
Question:
How many scenes are there in King John?.
"""


SYSTEM_PROMPT = """You are a SQL expert. Your task is to answer the user’s question by generating a final SQL query.
The schema of the database is provided in the user question.
You have access to a tool called `query_db` that you can use to run exploratory queries on the database. Use it only if needed to check values and validate assumptions.
When you perform exploratory queries, use LIMIT 3 if the output of the query may have more than 3 rows.
Eventually, provide ONLY the final SQL query enclosed within the tags <answer> and </answer>, not the answer to the query.
"""

In [15]:
from litellm import completion
import json
import re

pred_query = None
model_name = "hosted_vllm/Qwen/Qwen3-1.7B"
MAX_TURNS = 10 # a maximum number of turns in order to avoid infinite loops

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_QUESTION}
]

completion_params = {
    "model": model_name,
    "temperature": 0.7,
    "top_p": 0.8,
    "min_p": 0,
    "top_key": 20,
    "max_tokens": 4096,
    "repetition_penalty": 1.05,
    "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    "api_base": "http://127.0.0.1:8000/v1",
    "tools": TOOLS
}

print(USER_QUESTION)

for i in range(MAX_TURNS):

    # call the model
    response = completion(messages=messages, **completion_params)
    response_message = response.choices[0].message

    # add the model's response (thought + tool_call) to the history
    messages.append(response_message.model_dump())

    # check if the model has called a tool
    tool_calls = response_message.get("tool_calls", None)
    if tool_calls:
        # execution of the tool call
        for tool_call in tool_calls:
            call_id = tool_call["id"]
            if fn_call := tool_call.get("function"):
                fn_name = fn_call["name"]
                fn_args = json.loads(fn_call["arguments"])
                sql_query = fn_args.get("sql", "")
                pred_query = sql_query.strip()
                fn_res = get_function_by_name(fn_name)(**fn_args)

                #add the tool result (as JSON) to the history for the model
                messages.append({
                    "role": "tool",
                    "content": json.dumps({
                        "query_executed": sql_query,
                        "result": fn_res
                    }),
                    "tool_call_id": call_id,
                })

    # Check if the model is finished (gave a final answer without tool calls)
    elif response_message.content:
        final_output = response_message.content.strip()
        # Estrai il contenuto tra i tag <answer>
        match = re.search(r"<answer>\s*(.*?)\s*</answer>", final_output, re.DOTALL)
        if match:
          pred_query = match.group(1).strip()
          print(f"predicted query:\n{pred_query}")
        else:
          print("No <answer> tags found in final_output.")
        print(f"Final SQL query:\n{pred_query}")

        break


conn.close()


Database Schema:
CREATE TABLE "chapters"
(
    id          INTEGER
        primary key autoincrement,
    Act         INTEGER not null,
    Scene       INTEGER not null,
    Description TEXT    not null,
    work_id     INTEGER not null
        references works
);
CREATE TABLE "characters"
(
    id          INTEGER
        primary key autoincrement,
    CharName    TEXT not null,
    Abbrev      TEXT not null,
    Description TEXT not null
);
CREATE TABLE "paragraphs"
(
    id           INTEGER
        primary key autoincrement,
    ParagraphNum INTEGER           not null,
    PlainText    TEXT              not null,
    character_id INTEGER           not null
        references characters,
    chapter_id   INTEGER default 0 not null
        references chapters
);
CREATE TABLE "works"
(
    id        INTEGER
        primary key autoincrement,
    Title     TEXT    not null,
    LongTitle TEXT    not null,
    Date      INTEGER not null,
    GenreType TEXT    not null
);

Evidence:
Kin

In [16]:
messages

[{'role': 'system',
  'content': 'You are a SQL expert. Your task is to answer the user’s question by generating a final SQL query.\nThe schema of the database is provided in the user question.\nYou have access to a tool called `query_db` that you can use to run exploratory queries on the database. Use it only if needed to check values and validate assumptions.\nWhen you perform exploratory queries, use LIMIT 3 if the output of the query may have more than 3 rows.\nEventually, provide ONLY the final SQL query enclosed within the tags <answer> and </answer>, not the answer to the query.\n'},
 {'role': 'user',
  'content': '\nDatabase Schema:\nCREATE TABLE "chapters"\n(\n    id          INTEGER\n        primary key autoincrement,\n    Act         INTEGER not null,\n    Scene       INTEGER not null,\n    Description TEXT    not null,\n    work_id     INTEGER not null\n        references works\n);\nCREATE TABLE "characters"\n(\n    id          INTEGER\n        primary key autoincrement,\n 

In [17]:
output_string = ""
temp_parts = []

def clean_and_format(text):
    if text is None:
        return ''
    cleaned = text.replace('\n', ' ').replace('"', "'")
    cleaned = ' '.join(cleaned.split())
    return cleaned

for message in messages:

    # reasoning(se presente)
    reasoning = ""

    if 'provider_specific_fields' in message:
        reasoning = message['provider_specific_fields'].get('reasoning_content', "")

    if not reasoning:
        reasoning = message.get('reasoning_content', "")

    if reasoning:
        temp_parts.append(" | RAGIONAMENTO: " + clean_and_format(reasoning))

    # TOOL CALLS
    if message.get('tool_calls'):
        for tool_call in message['tool_calls']:
            try:
                args = json.loads(tool_call['function']['arguments'])
                sql_query = args.get('sql', 'N/A')
                temp_parts.append(" | CALL: " + clean_and_format(sql_query))
            except Exception:
                temp_parts.append(" | CALL: Errore nel parsing degli argomenti")

    # RISPOSTA DEL TOOL
    if message.get('role') == 'tool':
        try:
            obj = json.loads(message.get('content', '{}'))
            result = obj.get('result', 'Nessun Risultato Trovato')
            result_str = json.dumps(result)
            temp_parts.append(" | ANS: " + clean_and_format(result_str))
        except Exception:
            temp_parts.append(" | ANS: Errore nel parsing della risposta")

    # RISPOSTA FINALE DELL’ASSISTANT (solo se NON contiene tool calls)
    if (
        message.get('role') == 'assistant'
        and message.get('content')
        and not message.get('tool_calls')
    ):
        content = message['content'].strip()
        if content:
            temp_parts.append(" | RISPOSTA FINALE: " + clean_and_format(content))

# unisco tutto
output_string = " ".join(temp_parts).strip()
print(output_string)


| CALL: SELECT id FROM works WHERE Title = 'King John';  | ANS: [{'id': 17}]  | CALL: SELECT id, Act, Scene FROM chapters WHERE work_id = 17;  | ANS: [{'id': 19077, 'Act': 1, 'Scene': 1}, {'id': 19078, 'Act': 2, 'Scene': 1}, {'id': 19079, 'Act': 3, 'Scene': 1}, {'id': 19080, 'Act': 3, 'Scene': 2}, {'id': 19081, 'Act': 3, 'Scene': 3}, {'id': 19082, 'Act': 3, 'Scene': 4}, {'id': 19083, 'Act': 4, 'Scene': 1}, {'id': 19084, 'Act': 4, 'Scene': 2}, {'id': 19085, 'Act': 4, 'Scene': 3}, {'id': 19086, 'Act': 5, 'Scene': 1}, {'id': 19087, 'Act': 5, 'Scene': 2}, {'id': 19088, 'Act': 5, 'Scene': 3}, {'id': 19089, 'Act': 5, 'Scene': 4}, {'id': 19090, 'Act': 5, 'Scene': 5}, {'id': 19091, 'Act': 5, 'Scene': 6}, {'id': 19092, 'Act': 5, 'Scene': 7}]  | CALL: SELECT COUNT(DISTINCT Scene) FROM chapters WHERE work_id = 17;  | ANS: [{'COUNT(DISTINCT Scene)': 7}]  | RISPOSTA FINALE: The total number of scenes in 'King John' is 7. <answer> SELECT COUNT(DISTINCT Scene) FROM chapters WHERE work_id = 17; </answ

In [18]:
def compute_execution_accuracy(gt_results, predict_results):
  num_correct = 0
  num_queries = len(gt_results)
  mismatch_idx = []

  for i, result in enumerate(gt_results):
      if set(result['results']) == set(predict_results[i]['results']):
          num_correct += 1
      else:
          mismatch_idx.append(i)

  acc = (num_correct / num_queries) * 100

  return acc

In [20]:
def run_query(db_path, query):
  conn = sqlite3.connect(db_path)
  cursor = conn.cursor()
  cursor.execute(query)
  rows = cursor.fetchall()
  conn.close()

  # Flatten results and convert to list of strings
  return [row[0] for row in rows]

In [21]:
gt_query = """SELECT COUNT(*) AS n_Scenes FROM works w JOIN chapters c ON w.id = c.work_id WHERE w.Title = 'King John';
"""

In [22]:
pred_query

'SELECT COUNT(DISTINCT Scene) FROM chapters WHERE work_id = 17;'

In [23]:
run_query(db_name, pred_query)

[7]

In [24]:
rows_gt = run_query(db_name, gt_query)
gt_res = [{"results": rows_gt}]

rows_pred = run_query(db_name, pred_query)
pred_res = [{"results": rows_pred}]

In [25]:
acc = compute_execution_accuracy(gt_res, pred_res)
print(f"Accuracy of the generated SQL query: {acc}")

Accuracy of the generated SQL query: 0.0
